# Feature Engineering on High Volume For-Hire Vehicle (HVFHV) Trip Records and Demand Dataset:

In this notebook, we will aggregrated HVFHV dataset hourly and merge it with hourly demand dataset.

----

# Import Libraries:

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F 
from pyspark.sql.functions import * 
from pyspark.sql.functions import col, avg, count
import os

In [2]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("feature_engineering_demand+hvfhv")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.network.timeout", "600s")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.rpc.askTimeout", "600s")
    .config("spark.driver.memory", "100G")
    .config("spark.executor.memory", "100G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)

24/08/24 16:27:26 WARN Utils: Your hostname, Cocos-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 172.16.33.67 instead (on interface en0)
24/08/24 16:27:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/24 16:27:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Read Files:

In [3]:
base_dir = "../data"

Full HVFHV dataset:

In [4]:
full_hvfhv_sdf_dir = base_dir + '/developed/merged_data/full_hvfhv'
full_hvfhv_sdf = spark.read.parquet(full_hvfhv_sdf_dir)
full_hvfhv_sdf.show(5)

+-----------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+-----------+-------------------------+-------------------+-----------------+------------------+-----------+-----------+------------+------------+-----+
|hvfhs_license_num|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|wav_request_flag|wav_match_flag|day_of_week|request_to_pickup_minutes|         trip_speed|total_fare_amount|     total_revenue|pickup_date|pickup_hour|dropoff_date|dropoff_hour|month|
+-----------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+-----------+------

# Create the Utilization Rate by HVFHV License Number Dataset:

According to TLC driver income rules, the utilization rate is the percentage of time that high-volume drivers are transporting passengers. It is calculated by dividing the time spent with a passenger by the total time that drivers are logged into the app, including time waiting for a dispatch, time en route to pick up a passenger, and time with a passenger.

Here we can only calculate the rough utilization rate since we do not have the information about the time spent waiting for a dispatch in each order. So it is calculated by `trip_time`/(`request_to_pickup_minutes`+`trip_time`).

In [5]:
full_hvfhv_sdf = full_hvfhv_sdf.filter(full_hvfhv_sdf.request_to_pickup_minutes >= 0)

In [6]:
# Add new column `utilization_rate`
full_hvfhv_sdf = full_hvfhv_sdf.withColumn(
    'utilization_rate',
    col('trip_time') / (col('request_to_pickup_minutes') + col('trip_time'))
)

full_hvfhv_sdf.show(5)

+-----------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+-----------+-------------------------+-------------------+-----------------+------------------+-----------+-----------+------------+------------+-----+-------------------+
|hvfhs_license_num|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|wav_request_flag|wav_match_flag|day_of_week|request_to_pickup_minutes|         trip_speed|total_fare_amount|     total_revenue|pickup_date|pickup_hour|dropoff_date|dropoff_hour|month|   utilization_rate|
+-----------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------

In [7]:
full_hvfhv_sdf = full_hvfhv_sdf.filter((full_hvfhv_sdf.utilization_rate >= 0) &
                                       (full_hvfhv_sdf.utilization_rate <= 1))

In [8]:
num_rows = full_hvfhv_sdf.count()
print(f"Number of rows: {num_rows}")

columns = full_hvfhv_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

Number of rows: 99692683
Number of columns: 28


In [9]:
# Group by `pickup_hour` and `hvfhs_license_num`
# then calculate the average of `utilization_rate`
utilization_rate_sdf = full_hvfhv_sdf.groupBy(
    "pickup_hour", 
    "hvfhs_license_num"
).agg(
    avg("utilization_rate").alias("avg_utilization_rate")
).orderBy(
    "pickup_hour", 
    "hvfhs_license_num"
)

utilization_rate_sdf.show(5)

+-----------+-----------------+--------------------+
|pickup_hour|hvfhs_license_num|avg_utilization_rate|
+-----------+-----------------+--------------------+
|          0|                0|  0.7581537669544819|
|          0|                1|  0.7470081596734622|
|          1|                0|  0.7591253216764517|
|          1|                1|  0.7556135942119842|
|          2|                0|  0.7518118045863963|
+-----------+-----------------+--------------------+
only showing top 5 rows



# Aggregated Hourly Demand Dataset:

In [10]:
# Group by `pickup_hour`, `pickup_date``, `day_type`,
# then count the number of records for each group
hourly_demand_sdf = full_hvfhv_sdf.groupBy("pickup_hour", "pickup_date", "PULocationID") \
                                    .count() \
                                    .withColumnRenamed("count", "hourly_demand")\
                                    .withColumnRenamed("pickup_hour", "hour")\
                                    .withColumnRenamed("pickup_date", "date")\
                                    .withColumnRenamed("PULocationID", "location")
hourly_demand_sdf.show(5)

+----+----------+--------+-------------+
|hour|      date|location|hourly_demand|
+----+----------+--------+-------------+
|  17|2023-08-18|     246|          316|
|  17|2023-08-18|      13|          166|
|  17|2023-08-18|     159|          147|
|  18|2023-08-18|     137|          220|
|  19|2023-08-18|     159|          127|
+----+----------+--------+-------------+
only showing top 5 rows



# Aggregated Full HVFHV Dataset by Hour:

In [11]:
COLS = ["trip_miles", "trip_time", "utilization_rate", "base_passenger_fare", 
        "total_fare_amount", "tolls", "bcf", "sales_tax", "congestion_surcharge", 
        "airport_fee", "tips", "driver_pay", "shared_request_flag", 
        "shared_match_flag", "wav_request_flag", "wav_match_flag", 
        "request_to_pickup_minutes", "trip_speed", "total_revenue"]

# Drop unused columns
full_hvfhv_sdf = full_hvfhv_sdf.drop('hvfhs_license_num', 'DOLocationID', 
                                     'dropoff_date', 'dropoff_hour', )

# Create a list of aggregation expressions
agg_exprs = [F.avg(col).alias(f'avg_{col}') for col in COLS]

# Perform the aggregation
hourly_full_hvfhv_sdf = full_hvfhv_sdf.groupBy('pickup_date', 'day_of_week', 'month', 'pickup_hour', 'PULocationID').agg(*agg_exprs)

hourly_full_hvfhv_sdf.show(5)

+-----------+-----------+-----+-----------+------------+------------------+------------------+--------------------+-----------------------+---------------------+-------------------+-------------------+------------------+------------------------+--------------------+-------------------+------------------+-----------------------+---------------------+--------------------+--------------------+-----------------------------+-------------------+------------------+
|pickup_date|day_of_week|month|pickup_hour|PULocationID|    avg_trip_miles|     avg_trip_time|avg_utilization_rate|avg_base_passenger_fare|avg_total_fare_amount|          avg_tolls|            avg_bcf|     avg_sales_tax|avg_congestion_surcharge|     avg_airport_fee|           avg_tips|    avg_driver_pay|avg_shared_request_flag|avg_shared_match_flag|avg_wav_request_flag|  avg_wav_match_flag|avg_request_to_pickup_minutes|     avg_trip_speed| avg_total_revenue|
+-----------+-----------+-----+-----------+------------+------------------

In [12]:
# Add new column `day_type` to indicate it is weekday (0) or weekend (1)
hourly_full_hvfhv_sdf = hourly_full_hvfhv_sdf.withColumn(
    "day_type",
    F.when(hourly_full_hvfhv_sdf["day_of_week"].isin(["Monday", "Tuesday", "Wednesday", 
                                                      "Thursday", "Friday"]), 0)
    .otherwise(1)
)

# Drop unused columns
hourly_full_hvfhv_sdf = hourly_full_hvfhv_sdf.drop('day_of_week')

hourly_full_hvfhv_sdf.show(5)

+-----------+-----+-----------+------------+------------------+------------------+--------------------+-----------------------+---------------------+-------------------+-------------------+------------------+------------------------+--------------------+-------------------+------------------+-----------------------+---------------------+--------------------+--------------------+-----------------------------+-------------------+------------------+--------+
|pickup_date|month|pickup_hour|PULocationID|    avg_trip_miles|     avg_trip_time|avg_utilization_rate|avg_base_passenger_fare|avg_total_fare_amount|          avg_tolls|            avg_bcf|     avg_sales_tax|avg_congestion_surcharge|     avg_airport_fee|           avg_tips|    avg_driver_pay|avg_shared_request_flag|avg_shared_match_flag|avg_wav_request_flag|  avg_wav_match_flag|avg_request_to_pickup_minutes|     avg_trip_speed| avg_total_revenue|day_type|
+-----------+-----+-----------+------------+------------------+-----------------

In [13]:
num_rows = hourly_full_hvfhv_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hourly_full_hvfhv_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

Number of rows: 1108306
Number of columns: 24


In [14]:
hourly_full_hvfhv_sdf.describe().show()

+-------+-----------------+------------------+-----------------+------------------+-----------------+--------------------+-----------------------+---------------------+------------------+-------------------+------------------+------------------------+-------------------+------------------+------------------+-----------------------+---------------------+--------------------+-------------------+-----------------------------+-------------------+-----------------+-------------------+
|summary|            month|       pickup_hour|     PULocationID|    avg_trip_miles|    avg_trip_time|avg_utilization_rate|avg_base_passenger_fare|avg_total_fare_amount|         avg_tolls|            avg_bcf|     avg_sales_tax|avg_congestion_surcharge|    avg_airport_fee|          avg_tips|    avg_driver_pay|avg_shared_request_flag|avg_shared_match_flag|avg_wav_request_flag| avg_wav_match_flag|avg_request_to_pickup_minutes|     avg_trip_speed|avg_total_revenue|           day_type|
+-------+-----------------+---

# Create the Hourly Full HVFHV and Pickup Demand Related Dataset:

In [15]:
# Merge `hourly_demand_sdf` and `hourly_full_hvfhv_sdf` on hour and date
hourly_demand_hvfhv_sdf = hourly_demand_sdf.join(hourly_full_hvfhv_sdf, 
                                                 (hourly_full_hvfhv_sdf.pickup_hour == hourly_demand_sdf.hour) &
                                                 (hourly_full_hvfhv_sdf.pickup_date == hourly_demand_sdf.date) &
                                                 (hourly_full_hvfhv_sdf.PULocationID == hourly_demand_sdf.location), 
                                                 how='left').drop(hourly_demand_sdf['date']).drop(hourly_demand_sdf['location'])

hourly_demand_hvfhv_sdf = hourly_demand_hvfhv_sdf.drop(hourly_demand_sdf['hour'])
hourly_demand_hvfhv_sdf.show(5)

+-------------+-----------+-----+-----------+------------+------------------+------------------+--------------------+-----------------------+---------------------+------------------+------------------+------------------+------------------------+--------------------+-------------------+------------------+-----------------------+---------------------+--------------------+-------------------+-----------------------------+-------------------+------------------+--------+
|hourly_demand|pickup_date|month|pickup_hour|PULocationID|    avg_trip_miles|     avg_trip_time|avg_utilization_rate|avg_base_passenger_fare|avg_total_fare_amount|         avg_tolls|           avg_bcf|     avg_sales_tax|avg_congestion_surcharge|     avg_airport_fee|           avg_tips|    avg_driver_pay|avg_shared_request_flag|avg_shared_match_flag|avg_wav_request_flag| avg_wav_match_flag|avg_request_to_pickup_minutes|     avg_trip_speed| avg_total_revenue|day_type|
+-------------+-----------+-----+-----------+------------+

In [16]:
num_rows = hourly_demand_hvfhv_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hourly_demand_hvfhv_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

Number of rows: 1108306
Number of columns: 25


In [17]:
hourly_demand_hvfhv_sdf.printSchema()

root
 |-- hourly_demand: long (nullable = false)
 |-- pickup_date: date (nullable = true)
 |-- month: integer (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- avg_trip_miles: double (nullable = true)
 |-- avg_trip_time: double (nullable = true)
 |-- avg_utilization_rate: double (nullable = true)
 |-- avg_base_passenger_fare: double (nullable = true)
 |-- avg_total_fare_amount: double (nullable = true)
 |-- avg_tolls: double (nullable = true)
 |-- avg_bcf: double (nullable = true)
 |-- avg_sales_tax: double (nullable = true)
 |-- avg_congestion_surcharge: double (nullable = true)
 |-- avg_airport_fee: double (nullable = true)
 |-- avg_tips: double (nullable = true)
 |-- avg_driver_pay: double (nullable = true)
 |-- avg_shared_request_flag: double (nullable = true)
 |-- avg_shared_match_flag: double (nullable = true)
 |-- avg_wav_request_flag: double (nullable = true)
 |-- avg_wav_match_flag: double (nullable = true)
 |-- avg

# Save the Merged Datasets:

Utilization rate dataset:

In [11]:
utilization_rate_sdf_dir = base_dir + '/developed/merged_data'
file_name = 'utilization_rate'
utilization_rate_sdf_path = os.path.join(utilization_rate_sdf_dir, file_name)
utilization_rate_sdf.write.mode('overwrite').parquet(utilization_rate_sdf_path)

Hourly demand HVFHV dataset:

In [19]:
hourly_demand_hvfhv_sdf_dir = base_dir + '/developed/merged_data'
file_name = 'hourly_demand_hvfhv'
hourly_demand_hvfhv_sdf_path = os.path.join(hourly_demand_hvfhv_sdf_dir, file_name)
hourly_demand_hvfhv_sdf.write.mode('overwrite').parquet(hourly_demand_hvfhv_sdf_path)